# Pipeline Project

You will be using the provided data to create a machine learning model pipeline.

You must handle the data appropriately in your pipeline to predict whether an
item is recommended by a customer based on their review.
Note the data includes numerical, categorical, and text data.

You should ensure you properly train and evaluate your model.

## The Data

The dataset has been anonymized and cleaned of missing values.

There are 8 features for to use to predict whether a customer recommends or does
not recommend a product.
The `Recommended IND` column gives whether a customer recommends the product
where `1` is recommended and a `0` is not recommended.
This is your model's target/

The features can be summarized as the following:

- **Clothing ID**: Integer Categorical variable that refers to the specific piece being reviewed.
- **Age**: Positive Integer variable of the reviewers age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Positive Integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product high level division.
- **Department Name**: Categorical name of the product department name.
- **Class Name**: Categorical name of the product class name.

The target:
- **Recommended IND**: Binary variable stating where the customer recommends the product where 1 is recommended, 0 is not recommended.

## Load Data

In [1]:
import pandas as pd

# Load data
df = pd.read_csv(
    'data/reviews.csv',
)

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 18442 entries, 0 to 18441
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing ID              18442 non-null  int64
 1   Age                      18442 non-null  int64
 2   Title                    18442 non-null  str  
 3   Review Text              18442 non-null  str  
 4   Positive Feedback Count  18442 non-null  int64
 5   Division Name            18442 non-null  str  
 6   Department Name          18442 non-null  str  
 7   Class Name               18442 non-null  str  
 8   Recommended IND          18442 non-null  int64
dtypes: int64(4), str(5)
memory usage: 1.3 MB


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses,0
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits,1


## Preparing features (`X`) & target (`y`)

In [2]:
data = df

# separate features from labels
X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

Labels: [0 1]
Features:


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits


In [3]:
# Split data into train and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    random_state=27,
)

# Your Work

## Data Exploration

Let's look at the numerical and categorical features first.

In [4]:
df['Clothing ID'].value_counts()

Clothing ID
1078    871
862     658
1094    651
1081    487
829     452
       ... 
1188      1
271       1
7         1
55        1
631       1
Name: count, Length: 531, dtype: int64

In [5]:
df['Age'].describe()

count    18442.000000
mean        43.383635
std         12.246264
min         18.000000
25%         34.000000
50%         41.000000
75%         52.000000
max         99.000000
Name: Age, dtype: float64

In [6]:
df['Positive Feedback Count'].describe()

count    18442.000000
mean         2.697484
std          5.942220
min          0.000000
25%          0.000000
50%          1.000000
75%          3.000000
max        122.000000
Name: Positive Feedback Count, dtype: float64

In [7]:
df['Division Name'].value_counts()

Division Name
General           11664
General Petite     6778
Name: count, dtype: int64

In [8]:
df['Department Name'].value_counts()

Department Name
Tops        8713
Dresses     5371
Bottoms     3184
Jackets      879
Intimate     188
Trend        107
Name: count, dtype: int64

In [9]:
df['Class Name'].value_counts()

Class Name
Dresses           5371
Knits             3981
Blouses           2587
Sweaters          1218
Pants             1157
Jeans              970
Fine gauge         927
Skirts             796
Jackets            598
Outerwear          281
Shorts             260
Lounge             188
Trend              107
Casual bottoms       1
Name: count, dtype: int64

### Intermediate Resume

*Division Name* doesn't seem particularly useful with just two values. On the other hand, *Clothing ID* seems very relevant: good items would tend to get more favorable reviews and thus be recommended more often than bad items. Let's test this out by just looking at the DataFrame *Recommended IND* and *Positive Feedback Count* grouped by relevant *Clothing ID*.

In [10]:
most_reviewed_clothing_ids = df['Clothing ID'].value_counts()[:5]

df[df['Clothing ID'].isin(most_reviewed_clothing_ids.index)].groupby('Clothing ID')[['Recommended IND', 'Positive Feedback Count']].describe()

Recommended IND                                               \
                      count      mean       std  min  25%  50%  75%  max   
Clothing ID                                                                
829                   452.0  0.825221  0.380199  0.0  1.0  1.0  1.0  1.0   
862                   658.0  0.811550  0.391368  0.0  1.0  1.0  1.0  1.0   
1078                  871.0  0.811711  0.391168  0.0  1.0  1.0  1.0  1.0   
1081                  487.0  0.852156  0.355310  0.0  1.0  1.0  1.0  1.0   
1094                  651.0  0.818740  0.385529  0.0  1.0  1.0  1.0  1.0   

            Positive Feedback Count                                          \
                              count      mean       std  min  25%  50%  75%   
Clothing ID                                                                   
829                           452.0  2.457965  4.541067  0.0  0.0  1.0  3.0   
862                           658.0  2.583587  5.362918  0.0  0.0  0.5  2.0   
1078                          871.0  2.859931  7.288229  0.0  0.0  1.0  3.0   
1081                          487.0  3.203285  7.379885  0.0  0.0  1.0  3.0   
1094                          651.0  3.480799  7.883431  0.0  0.0  1.0  3.0   

                   
              max  
Clothing ID        
829          44.0  
862          43.0  
1078         98.0  
1081         81.0  
1094         89.0

It seems our intuition was correct: good items tend to be recommended a lot. Similarly, we might expect that people who took the time to write the first review tend to recommend the product, because anecdotally, people don't tend to waste further time on bad items with the possible exception to warn others if the items is truly terrible.

In [11]:
least_reviewed_clothing_ids = df['Clothing ID'].value_counts()[-5:]
df[df['Clothing ID'].isin(least_reviewed_clothing_ids.index)].groupby('Clothing ID')[['Recommended IND', 'Positive Feedback Count']].describe()

Recommended IND                                    \
                      count mean std  min  25%  50%  75%  max   
Clothing ID                                                     
7                       1.0  1.0 NaN  1.0  1.0  1.0  1.0  1.0   
55                      1.0  1.0 NaN  1.0  1.0  1.0  1.0  1.0   
271                     1.0  1.0 NaN  1.0  1.0  1.0  1.0  1.0   
631                     1.0  1.0 NaN  1.0  1.0  1.0  1.0  1.0   
1188                    1.0  1.0 NaN  1.0  1.0  1.0  1.0  1.0   

            Positive Feedback Count                                          
                              count  mean std   min   25%   50%   75%   max  
Clothing ID                                                                  
7                               1.0   0.0 NaN   0.0   0.0   0.0   0.0   0.0  
55                              1.0   0.0 NaN   0.0   0.0   0.0   0.0   0.0  
271                             1.0  13.0 NaN  13.0  13.0  13.0  13.0  13.0  
631                             1.0   0.0 NaN   0.0   0.0   0.0   0.0   0.0  
1188                            1.0   0.0 NaN   0.0   0.0   0.0   0.0   0.0

Let's see if we can find at least one terrible item, where the single reviewer went out of their way to warn others.

In [12]:
# let's get value counts
value_counts = df['Clothing ID'].value_counts()
# limit to the first 100 with 1 review
single_review_clothing_ids = value_counts[value_counts == 1][:100]

single_review_clothing = df[df['Clothing ID'].isin(single_review_clothing_ids.index)]

single_review_clothing['Recommended IND'].value_counts()

Recommended IND
1    83
0    17
Name: count, dtype: int64

Let's look for a few example terrible items.

In [13]:
single_review_clothing[single_review_clothing['Recommended IND'] == 0]

,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
1567,63,33,Extra fabric,There is a lot of fabric here! the legs fit me...,0,General,Bottoms,Shorts,0
2004,18,42,Disappointing quality,"I ordered these leggings and loved them, for a...",0,General Petite,Bottoms,Jeans,0
2232,429,23,Pretty but poor quality,When these shorts arrived i loved them and was...,3,General,Bottoms,Shorts,0
3258,1064,46,I wanted to love it.,I read the other reviews before i purchased th...,0,General,Bottoms,Pants,0
5625,1023,57,Beautiful but runs huge,I was so glad to find such a beautiful item ma...,2,General,Bottoms,Jeans,0
5808,894,41,Disapppointed,Was excited about this sweater but it really d...,1,General,Tops,Fine gauge,0
9194,693,51,Awkward fit,I was so excited to try these on at my local r...,0,General Petite,Intimate,Lounge,0
9511,734,39,Beautiful but sheer,These pajama pants are beautiful but very shee...,0,General Petite,Intimate,Lounge,0
10768,669,21,Pretty but cheap,These are light and airy and pretty. they also...,6,General Petite,Intimate,Lounge,0
11263,1073,37,Sad this couldn't come home with me....,So i really loved this dress. i am tall (6ft) ...,7,General Petite,Dresses,Dresses,0


## Building Pipeline

First, determine which features are *text*, requiring natural language processing (NLP), *categorical*, requiring encoding, or *numerical* requiring scaling.

In [14]:
text_features = ('Title', 'Review Text')
cat_features = ('Clothing ID', 'Division Name', 'Department Name', 'Class Name')
num_features = ('Positive Feedback Count', 'Age')

Next, let's build a few basic classes to assist in natural language processing (NLP).

In [15]:
from collections.abc import Collection
from sklearn.base import BaseEstimator, TransformerMixin
import spacy

class CharCounter(BaseEstimator, TransformerMixin):

    def __init__(self, character: str):
        """Character Counter Class

        Args:
            character: character to count in the text.
        """
        self.character = character

    def fit(self, X, y=None):
        """Fit to Data (NOP)"""
        self.is_fitted_ = True
        return self

    def transform(self, X):
        """Transform Data
        
        Args:
            X: array-like of shape (n_samples, n_features) input samples.
        
        Returns:
            array-like of shape (n_samples, n_features) transformed samples.
        """
        return [[x.count(self.character)] for x in X]

class SentimentAnalysis(BaseEstimator, TransformerMixin):
    def __init__(self, nlp: spacy.Language, pos: Collection[str]):
        """Sentiment Analysis
        
        Args:
            nlp: spacy Language model.
            pos: collection of part-of-speech tags to consider for sentiment
                 analysis, e.g. 'NOUN', 'ADJ' etc.
        """
        self.nlp = nlp
        self.pos = pos
    
    def fit(self, X, y=None):
        """Fit to Data (NOP)"""
        self.is_fitted_ = True
        return self

    def transform(self, X):
        """Transform Data
        
        Args:
            X: array-like of shape (n_samples, n_features) input samples.
        
        Returns:
            list[str]: one lemmatized document string per sample.
        """
        return [
            ' '.join(token.lemma_ for token in doc if token.pos_ in self.pos)
            for doc in self.nlp.pipe(X)
        ]

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import FeatureUnion, Pipeline
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    import numpy as np

# load English Core Web Small model
nlp = spacy.load('en_core_web_sm')

def _combine_text(X: pd.DataFrame) -> 'np.ndarray':
    """Combine text features into a single feature.

    Args:
        X: DataFrame with text features.

    Returns:
        np.ndarray: Combined text features.
    """
    return X.fillna("").astype(str).agg(' '.join, axis=1).to_numpy()

features = ColumnTransformer([
    ('txt', Pipeline([
        ('combine',  FunctionTransformer(_combine_text, validate=False)),
        ('features', FeatureUnion([
            ('chars',       CharCounter(character='')), # technically, off-by-one '' => 1 etc.
            ('spaces',      CharCounter(character=' ')),
            ('exclamation', CharCounter(character='!')),
            ('question',    CharCounter(character='?')),
            ('sentiment',   Pipeline([
                ('nlp',         SentimentAnalysis(nlp, ('NOUN', 'VERB', 'ADJ', 'ADV'))),
                ('tf-idf',      TfidfVectorizer()),
            ])),
        ])),
    ]), text_features),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_features),
    #('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features),
    ('num', MinMaxScaler(feature_range=(0, 1), clip=True), num_features),
])

## Training Pipeline

In [17]:
from sklearn.ensemble import RandomForestClassifier

parameters = {
    'n_estimators': 100,
    'criterion':    'gini',
    'max_depth':    None,
    'max_features': 'sqrt',
    'max_samples':  None,
    'random_state':  0x2f905959,
    'n_jobs':       -1,
    'verbose':      False,
}

model = Pipeline([
    ('features', features),
    ('classifier', RandomForestClassifier(**parameters)),
])

In [18]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Clothing ID','Age','Title',...,'Division Name','Department Name', 'Class Name']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('txt', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passth

Evaluate trained model for accuracy:

In [19]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print('Accuracy:', accuracy)

Accuracy: 0.851490514905149


That's ~85.15% accuracy on first try, which seems really good.

## Fine-Tuning Pipeline

Let's now perform a grid search over relevant Random Forest and pre-processing pipeline parameters.

In [20]:
list(model.get_params().keys())

['memory',
 'steps',
 'transform_input',
 'verbose',
 'features',
 'classifier',
 'features__n_jobs',
 'features__remainder',
 'features__sparse_threshold',
 'features__transformer_weights',
 'features__transformers',
 'features__verbose',
 'features__verbose_feature_names_out',
 'features__txt',
 'features__cat',
 'features__num',
 'features__txt__memory',
 'features__txt__steps',
 'features__txt__transform_input',
 'features__txt__verbose',
 'features__txt__combine',
 'features__txt__features',
 'features__txt__combine__accept_sparse',
 'features__txt__combine__check_inverse',
 'features__txt__combine__feature_names_out',
 'features__txt__combine__func',
 'features__txt__combine__inv_kw_args',
 'features__txt__combine__inverse_func',
 'features__txt__combine__kw_args',
 'features__txt__combine__validate',
 'features__txt__features__n_jobs',
 'features__txt__features__transformer_list',
 'features__txt__features__transformer_weights',
 'features__txt__features__verbose',
 'features__t

In [23]:
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV

estimator = clone(model)

params = dict(
    features__txt__features__sentiment__nlp__pos = [('NOUN', 'VERB', 'ADJ', 'ADV'), ('NOUN', 'ADJ'), ('VERB', 'ADV'), ('ADJ', 'ADV')],
    classifier__criterion = ['gini', 'entropy', 'log_loss'],
    classifier__n_estimators = [100, 150, 200, 250],
    classifier__max_depth = [None, 50, 100, 150],
    classifier__max_features = ['sqrt', 'log2', 0.5, 0.75, 1.0],
    classifier__max_samples = [None, 0.5, 0.75, 1.0],
)

search = RandomizedSearchCV(
    estimator,
    params,
    n_iter=10,    # Try 10 different combinations of parameters
    cv=3,         # Use 3-fold cross-validation
    refit=True,   # Refit the model using the best parameters found
    n_jobs=-1,    # Use all available processors (for multiprocessing)
    verbose=3,    # Output of parameters, score, time
    random_state=0x2f905959,
)
search.fit(X_train, y_train)

search.best_params_

Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'features__txt__features__sentiment__nlp__pos': ('NOUN',
  'VERB',
  'ADJ',
  'ADV'),
 'classifier__n_estimators': 150,
 'classifier__max_samples': 1.0,
 'classifier__max_features': 0.5,
 'classifier__max_depth': 100,
 'classifier__criterion': 'gini'}

In [ ]:
y_pred_best = search.best_estimator_.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)

print('Accuracy:', accuracy_best)

Accuracy: 0.8737127371273713


Well, ~87.37% is a modest improvement over the baseline.

In [24]:
from pathlib import Path
import joblib

filename = Path('model.joblib')
joblib.dump(model, filename )

print(f'Model saved to {filename}')

Model saved to model.joblib
